# Паттерн 7: Plan-and-Execute — планировщик + исполнители

Паттерн Plan-and-Execute разделяет стратегическое мышление и тактическое выполнение. Планировщик думает, исполнитель делает, перепланировщик корректирует. Три агента, каждый со своей ролью: planner создаёт структурированный план из нескольких шагов, executor выполняет один шаг за раз (с контекстом предыдущих результатов), replanner оценивает прогресс и решает — продолжить или завершить. Финализатор синтезирует итоговый ответ из всех результатов.

```mermaid
graph LR
    START((START)) --> planner{{"🎯 Planner<br/><i>составляет план</i>"}}
    planner --> executor(["🔹 Executor<br/><i>выполняет шаг</i>"])
    executor --> replanner{{"🎯 Replanner<br/><i>корректирует план</i>"}}
    replanner -->|continue| executor
    replanner -->|done| finalizer(["🔹 Finalizer<br/><i>собирает результат</i>"])
    finalizer --> END((END))

    classDef coordinator fill:#4A90D9,stroke:#2C5F8A,color:#fff,stroke-width:2px
    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px
    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff

    class planner,replanner coordinator
    class executor,finalizer worker
    class START,END terminal
```

In [1]:
import sys
sys.path.insert(0, "..")

import json
import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from llm_config import get_llm, check_api_key

In [2]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


## Состояние плана

`PlanState` содержит пять полей. `plan` — список шагов, созданных планировщиком. `completed` — выполненные шаги с редьюсером `operator.add`: каждый исполнитель возвращает список, и LangGraph автоматически добавляет его к существующему, а не перезаписывает. `current_step` — текущий шаг, который исполнитель должен выполнить. Планировщик заполняет `plan`, исполнитель переносит шаги из `plan` в `completed`, а `final_answer` заполняется финализатором в самом конце.

In [3]:
llm = get_llm()

class PlanState(TypedDict):
    goal: str                                          # Цель от пользователя
    plan: list[str]                                    # Список шагов плана
    completed: Annotated[list[str], operator.add]      # Выполненные шаги (аддитивный)
    current_step: str                                  # Текущий шаг для исполнителя
    final_answer: str                                  # Итоговый ответ

## Узел планировщика

Планировщик — мощная модель (для сложных задач можно использовать `get_llm("gpt-5.4")`), которая анализирует цель и создаёт структурированный план. Он возвращает JSON-список шагов. Поскольку LLM не всегда отвечает чистым JSON, предусмотрен fallback: при ошибке парсинга текст разбивается по строкам и фильтруется. После создания плана первый шаг устанавливается как текущий для исполнителя.

In [4]:
def planner_node(state: PlanState) -> dict:
    """
    Планировщик: анализирует цель и создаёт пошаговый план.
    Для сложных задач можно использовать более мощную модель:
    get_llm("gpt-5.4").
    """
    response = llm.invoke([
        SystemMessage(content="""Ты — планировщик задач. Создай чёткий план из 3-4 шагов
для достижения цели. Каждый шаг должен быть конкретным и выполнимым.

Верни JSON-список (и ТОЛЬКО JSON, без markdown-блоков):
["шаг 1", "шаг 2", "шаг 3"]

Пример:
["Найти ключевые факты", "Проанализировать тренды", "Написать обзор"]"""),
        HumanMessage(content=f"Цель: {state['goal']}")
    ])

    # Парсинг JSON с fallback
    try:
        plan = json.loads(response.content.strip())
        if not isinstance(plan, list):
            raise ValueError("Ожидался список")
    except (json.JSONDecodeError, ValueError):
        # Fallback: разбиваем текст по строкам
        lines = [
            line.strip().lstrip("0123456789.-) ")
            for line in response.content.split("\n")
            if line.strip()
        ]
        plan = [l for l in lines if len(l) > 5][:4]

    print(f"  [Планировщик] План из {len(plan)} шагов:")
    for i, step in enumerate(plan):
        print(f"    {i + 1}. {step}")

    # Устанавливаем первый шаг как текущий
    current = plan[0] if plan else "Выполнить задачу"
    return {"plan": plan, "current_step": current}

## Узел исполнителя

Исполнитель — быстрая модель (`get_llm()` по умолчанию), которая выполняет один шаг за раз. Он видит контекст предыдущих выполненных шагов, что позволяет каждому следующему шагу опираться на результаты предыдущих. В реальной системе исполнитель мог бы быть полноценным агентом с инструментами: поисковыми API, калькулятором, интерпретатором Python. Результат записывается в `completed` через редьюсер.

In [5]:
def executor_node(state: PlanState) -> dict:
    """
    Исполнитель: выполняет текущий шаг плана.
    Видит контекст предыдущих выполненных шагов.
    """
    step = state["current_step"]

    # Контекст: что уже выполнено
    context = ""
    if state["completed"]:
        context = "\n\nРезультаты предыдущих шагов:\n" + "\n".join(state["completed"])

    response = llm.invoke([
        SystemMessage(content=(
            "Ты — исполнитель задач. Выполни указанный шаг. "
            "Дай конкретный, полезный результат (3-4 предложения). "
            "Если есть результаты предыдущих шагов — используй их."
        )),
        HumanMessage(content=f"Общая цель: {state['goal']}\n\nТекущий шаг: {step}{context}")
    ])

    result = f"Шаг [{step}]: {response.content}"
    print(f"  [Исполнитель] «{step}» — выполнен")
    return {"completed": [result]}

## Узел перепланировщика

Перепланировщик анализирует прогресс выполнения: сравнивает количество выполненных шагов с общим числом шагов в плане. Если есть невыполненные шаги — устанавливает следующий шаг как текущий и возвращает управление к исполнителю. Если все шаги выполнены — устанавливает пустой `current_step`, что является сигналом для условного ребра направить управление к финализатору.

In [6]:
def replanner_node(state: PlanState) -> dict:
    """
    Перепланировщик: проверяет прогресс и определяет следующий шаг.
    Если все шаги выполнены — не устанавливает current_step.
    """
    completed_count = len(state["completed"])
    plan = state["plan"]
    total = len(plan)

    print(f"  [Перепланировщик] Прогресс: {completed_count}/{total}")

    if completed_count < total:
        next_step = plan[completed_count]
        print(f"  [Перепланировщик] Следующий шаг: {next_step}")
        return {"current_step": next_step}

    print(f"  [Перепланировщик] Все шаги выполнены → финализация")
    return {"current_step": ""}

## Узел финализатора

Финализатор — отдельный узел, а не часть перепланировщика. Это разделение ответственности: перепланировщик решает «что дальше», финализатор решает «как подытожить». Каждый делает одну вещь. Финализатор получает все результаты из `completed` и синтезирует связный итоговый ответ.

In [7]:
def finalizer_node(state: PlanState) -> dict:
    """
    Финализатор: синтезирует итоговый ответ из всех результатов.
    """
    results_text = "\n\n".join(state["completed"])

    response = llm.invoke([
        SystemMessage(content=(
            "Ты — финализатор. Синтезируй краткий, связный итоговый ответ "
            "(4-5 предложений) из результатов всех выполненных шагов. "
            "Не повторяй шаги — дай единый связный текст."
        )),
        HumanMessage(content=f"Цель: {state['goal']}\n\nРезультаты шагов:\n{results_text}")
    ])

    print(f"  [Финализатор] Итоговый ответ готов")
    return {"final_answer": response.content}

## Маршрутизация и сборка графа

Условное ребро после перепланировщика проверяет: есть ли ещё невыполненные шаги? Функция `should_continue` сравнивает длину `completed` с длиной `plan`. Если шаги остались — управление возвращается к исполнителю (`"execute"`). Если все выполнены — к финализатору (`"finalize"`). Цикл выполнения: `START -> planner -> executor -> replanner -> executor -> replanner -> ... -> finalizer -> END`.

In [8]:
def should_continue(state: PlanState) -> str:
    """Ещё есть шаги → executor, все выполнены → finalizer."""
    completed_count = len(state["completed"])
    total = len(state["plan"])
    if completed_count >= total:
        return "finalize"
    return "execute"

# --- Сборка графа ---
graph = StateGraph(PlanState)

graph.add_node("planner", planner_node)
graph.add_node("executor", executor_node)
graph.add_node("replanner", replanner_node)
graph.add_node("finalizer", finalizer_node)

graph.add_edge(START, "planner")
graph.add_edge("planner", "executor")
graph.add_edge("executor", "replanner")
graph.add_conditional_edges("replanner", should_continue, {
    "execute": "executor",
    "finalize": "finalizer",
})
graph.add_edge("finalizer", END)

app = graph.compile()

## Запуск

Передаём цель — и план создаётся автоматически. Планировщик разобьёт задачу на 3-4 шага, исполнитель выполнит каждый по очереди с контекстом предыдущих, а финализатор соберёт всё в итоговый ответ. Весь цикл виден в логах: какие шаги создал планировщик, что выполнил исполнитель, как перепланировщик отслеживал прогресс.

In [9]:
result = app.invoke({
    "goal": "Подготовь обзор трендов в AI-агентах и мультиагентных системах за 2025 год",
    "plan": [],
    "completed": [],
    "current_step": "",
    "final_answer": "",
})

print(f"\n  Шагов в плане: {len(result['plan'])}")
print(f"  Выполнено: {len(result['completed'])}")
print(f"  Итоговый ответ: {result['final_answer'][:200]}...")

  [Планировщик] План из 4 шагов:
    1. Собрать ключевые источники за 2025 год: отчёты, статьи, релизы крупных компаний и исследовательские публикации по AI-агентам и мультиагентным системам
    2. Сгруппировать материалы по темам: архитектуры агентов, инструменты и оркестрация, автономность, память и планирование, безопасность, оценки и бенчмарки, кейсы внедрения
    3. Выделить главные тренды 2025 года, сравнить их с 2024 годом и зафиксировать изменения в возможностях, ограничениях и сценариях применения
    4. Составить структурированный обзор с краткими выводами, примерами и практическими рекомендациями для бизнеса и разработки


  [Исполнитель] «Собрать ключевые источники за 2025 год: отчёты, статьи, релизы крупных компаний и исследовательские публикации по AI-агентам и мультиагентным системам» — выполнен
  [Перепланировщик] Прогресс: 1/4
  [Перепланировщик] Следующий шаг: Сгруппировать материалы по темам: архитектуры агентов, инструменты и оркестрация, автономность, память и планирование, безопасность, оценки и бенчмарки, кейсы внедрения


  [Исполнитель] «Сгруппировать материалы по темам: архитектуры агентов, инструменты и оркестрация, автономность, память и планирование, безопасность, оценки и бенчмарки, кейсы внедрения» — выполнен
  [Перепланировщик] Прогресс: 2/4
  [Перепланировщик] Следующий шаг: Выделить главные тренды 2025 года, сравнить их с 2024 годом и зафиксировать изменения в возможностях, ограничениях и сценариях применения


  [Исполнитель] «Выделить главные тренды 2025 года, сравнить их с 2024 годом и зафиксировать изменения в возможностях, ограничениях и сценариях применения» — выполнен
  [Перепланировщик] Прогресс: 3/4
  [Перепланировщик] Следующий шаг: Составить структурированный обзор с краткими выводами, примерами и практическими рекомендациями для бизнеса и разработки


  [Исполнитель] «Составить структурированный обзор с краткими выводами, примерами и практическими рекомендациями для бизнеса и разработки» — выполнен
  [Перепланировщик] Прогресс: 4/4
  [Перепланировщик] Все шаги выполнены → финализация


  [Финализатор] Итоговый ответ готов

  Шагов в плане: 4
  Выполнено: 4
  Итоговый ответ: В 2025 году рынок AI-агентов сместился от «чат-ботов с инструментами» к agentic systems, которые способны не только отвечать, но и выполнять цепочки действий, координировать шаги и брать на себя части...


## Итоги

Паттерн Plan-and-Execute реализует чёткое разделение ответственности между четырьмя узлами:

- **Планировщик** создаёт структурированный план — список конкретных шагов для достижения цели
- **Исполнитель** выполняет один шаг за раз, используя контекст всех предыдущих результатов
- **Перепланировщик** отслеживает прогресс и решает: вернуться к исполнителю или перейти к финализации
- **Финализатор** синтезирует единый связный ответ из всех промежуточных результатов

Ключевая особенность — цикл `executor -> replanner`, управляемый условным ребром. Это позволяет системе итеративно выполнять план шаг за шагом, сохраняя контекст. В продакшн-версии исполнитель может быть полноценным агентом с инструментами, а планировщик — использовать более мощную модель для стратегического мышления.